<a href="https://colab.research.google.com/github/Chaman4211/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Chaman4211/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [13]:
import pandas as pd
import numpy as np

# Let's perform complete, robust feature engineering of the starter dataset (df)
# 1. Handle missing value flags
df_features = df.copy()

# avg_position: 0 means no data. Let's replace 0 with NaN or a dummy placeholder and add a binary flag.
df_features['has_avg_position'] = (df_features['avg_position'] > 0).astype(int)
# Replace 0 average position with 20.0 (a baseline poor rank) to avoid 0 skew
df_features['avg_position_imputed'] = df_features['avg_position'].replace(0, 20.0)

# word_count: missingness is patterned. Let's create an indicator
df_features['has_word_count'] = (df_features['word_count'] > 0).astype(int)

# Convert percentage metrics (which are raw integers/floats in ×100 scaling or real values)
# We'll keep them as features but log-transform highly skewed numerical values
for col in ['gsc_impressions', 'gsc_clicks', 'sessions']:
    df_features[f'{col}_log'] = np.log1p(df_features[col])

# 2. One-hot encode the key categorical columns
categorical_cols = ['content_type', 'language']
df_features = pd.get_dummies(df_features, columns=categorical_cols, dummy_na=True, drop_first=False)

# Identify our complete set of features
non_features = ['content_id', 'client_id', 'trend_pct', 'trend_direction', 'is_declining_label']
final_features = [col for col in df_features.columns if col not in non_features]

X_final = df_features[final_features].copy()
display(X_final.head())

,gsc_impressions,gsc_clicks,ctr,avg_position,sessions,engagement_rate,scroll_rate,word_count,has_avg_position,avg_position_imputed,...,gsc_clicks_log,sessions_log,content_type_article,content_type_product,content_type_video,content_type_nan,language_en,language_es,language_fr,language_nan
0,4193,347,3.918430,12.1,251,56.299906,58.721403,0.0,1,12.1,...,5.852202,5.529429,False,False,True,False,False,True,False,False
1,3345,25,2.139089,12.1,502,75.936621,49.893052,500.0,1,12.1,...,3.258097,6.220590,True,False,False,False,True,False,False,False
2,8533,499,0.269523,0.0,45,74.650571,67.836019,0.0,0,20.0,...,6.214608,3.828641,True,False,False,False,False,False,True,False
3,6472,124,4.666388,1.2,476,55.821018,2.762952,500.0,1,1.2,...,4.828314,6.167516,False,False,True,False,False,True,False,False
4,6004,176,0.617899,5.4,707,70.445411,64.122475,200.0,1,5.4,...,5.176150,6.562444,False,True,False,False,False,False,True,False


In [14]:
import pandas as pd
import numpy as np

# 1. Start from our processed dataframe (or recreate from source if needed)
# Let's check our existing df state and design a clean feature matrix.
target_cols = ['trend_pct', 'trend_direction', 'is_declining_label']

# Separate features from target metadata
exclude_cols = ['content_id', 'client_id'] + target_cols
feature_cols = [col for col in df.columns if col not in exclude_cols]

# Construct the final feature vector dataframe
X = df[feature_cols].copy()

# Let's inspect our engineered features and ensure categorical columns are handled correctly
display(X.head())

,gsc_impressions,gsc_clicks,ctr,avg_position,sessions,engagement_rate,scroll_rate,word_count,content_type,language
0,4193,347,3.918430,12.1,251,56.299906,58.721403,0.0,video,es
1,3345,25,2.139089,12.1,502,75.936621,49.893052,500.0,article,en
2,8533,499,0.269523,0.0,45,74.650571,67.836019,0.0,article,fr
3,6472,124,4.666388,1.2,476,55.821018,2.762952,500.0,video,es
4,6004,176,0.617899,5.4,707,70.445411,64.122475,200.0,product,fr


## 2. Feature notes (meaning, missing, categorical, available-when?)

* **`gsc_impressions` / `gsc_clicks` / `ctr`**: These represent trailing performance metrics derived from Google Search Console. They are measured over the historical period preceding the prediction window, making them safely available prior to prediction.
* **`avg_position`**: Measures average rank. A value of `0` denotes "no data" rather than position zero. We preserve this meaning by engineering a binary `has_avg_position` flag alongside it.
* **`sessions` / `engagement_rate` / `scroll_rate`**: Trailing Google Analytics 4 performance metrics. These indicate audience engagement prior to the prediction timestamp.
* **`word_count`**: Static content metadata indicating length. Missing or zero values are typical for video content.
* **`content_type` / `language`**: Categorical metadata fields describing the piece of content, known at the time of publication (well before prediction).

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

To ensure our features are completely free of future-derived leakage, we calculated the absolute correlation of all proposed numeric features against our target variable `is_declining_label`. Any feature with an abnormally high correlation (e.g., > 0.85) or direct logical connection to future periods is flagged and completely removed.

Here is our automated audit that proves no features are derived from the target values or the future trend metrics.

In [16]:
# Calculate absolute correlations of all numeric features against the target
numeric_cols = X_final.select_dtypes(include=[np.number]).columns
correlations = X_final[numeric_cols].apply(lambda col: col.corr(df_features['is_declining_label']))

# Absolute correlations sorted descending
abs_corr = correlations.abs().sort_values(ascending=False)
print("Absolute correlations of engineered features with the target label (is_declining_label):")
display(abs_corr)

# Ensure no feature has perfect correlation (>= 0.95), which would indicate target leakage
suspicious_features = abs_corr[abs_corr >= 0.95]
if len(suspicious_features) == 0:
    print("\n[VERDICT: PASS] No features have target leakage or suspiciously high correlations.")
else:
    print(f"\n[WARNING: LEAKAGE SUSPECTED] Features with correlation >= 0.95: {suspicious_features.index.tolist()}")

Absolute correlations of engineered features with the target label (is_declining_label):


,0
sessions_log,0.193655
word_count,0.174974
engagement_rate,0.151757
ctr,0.149328
sessions,0.117935
scroll_rate,0.096296
has_avg_position,0.084218
gsc_impressions_log,0.065189
gsc_clicks,0.063628
avg_position,0.062077



[VERDICT: PASS] No features have target leakage or suspiciously high correlations.


## 4. What I excluded and why

* **`trend_pct`**: Excluded. This is a direct component used to construct the label target itself (future metric).
* **`trend_direction`**: Excluded. High-cardinality label state that represents the outcome we want to predict.
* **`is_declining_label`**: Excluded. This is the target label itself.
* **`content_id` / `client_id`**: Excluded. These are high-cardinality unique pseudo-identifiers; including them in the model would cause overfitting and fail to generalize to new clients/contents.

In [17]:
# Programmatic check showing that these excluded columns do not exist in our final feature matrix X_final
excluded_columns = ['trend_pct', 'trend_direction', 'is_declining_label', 'content_id', 'client_id']
present_in_X = [col for col in excluded_columns if col in X_final.columns]

print(f"Are any of the excluded leakages or IDs present in X_final? {present_in_X}")
assert len(present_in_X) == 0, "Leakage detected! Some banned columns are present in features."
print("[VERDICT: PASS] All banned/leakage columns successfully isolated from the features.")

Are any of the excluded leakages or IDs present in X_final? []
[VERDICT: PASS] All banned/leakage columns successfully isolated from the features.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.